In [1]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    !pip install -q pyspark mlflow seaborn pyyaml
    ROOT = Path('/content/drive/MyDrive/BSE/Big Data/Homework_Labs/'
                'Lab3-BigDataArchitectures/lab3-data-engineering')
except ImportError:
    ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent

assert (ROOT / 'src').exists(), f"src/ not found under {ROOT}"
sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ROOT = /content/drive/MyDrive/BSE/Big Data/Homework_Labs/Lab3-BigDataArchitectures/lab3-data-engineering


# B.1 + B.2: Training & MLflow

Trains the spark.ml classifiers, ranks by validation accuracy, registers and promotes
the best model, then reloads it from the registry to prove reproducibility.
Charts and B.3 discussion live in `validation.ipynb`.

In [4]:
import mlflow

from src.utils.config import load_config
from src.utils.spark_session import get_spark
from src.training import train

config = load_config()

## MLflow store

Model Registry needs a SQL backend. Train against a local SQLite store (no Drive-FUSE
lock errors), restore a prior store from Drive if present, sync back to Drive after.

In [5]:
import shutil

# Local SQLite store during the run; restore from Drive if a prior store exists.
drive_store = ROOT / "mlflow"
local_store = Path("/content/mlflow") if Path("/content").exists() else drive_store
if local_store != drive_store and drive_store.exists() and not local_store.exists():
    shutil.copytree(drive_store, local_store)
local_store.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(f"sqlite:///{local_store / 'mlflow.db'}")
experiment = config["mlflow"]["experiment"]
if mlflow.get_experiment_by_name(experiment) is None:
    mlflow.create_experiment(experiment, artifact_location=str(local_store / "artifacts"))
mlflow.set_experiment(experiment)

2026/06/24 13:15:28 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/24 13:15:29 INFO mlflow.store.db.utils: Updating database tables


<Experiment: artifact_location='/content/mlflow/artifacts', creation_time=1782306932008, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1782306932008, lifecycle_stage='active', name='barcelona_price_tier', tags={}, trace_location=None, workspace='default'>

## Train, rank, deploy best, reload from registry (B.1 + B.2)

Trains both classifiers, logs each run with the data fingerprint, registers and promotes
the best by validation accuracy, then reloads it from the registry and re-scores.

In [6]:
spark = get_spark(config)
summary = train(spark, config)

print("data_fingerprint:", summary["fingerprint"])
for r in summary["results"]:
    print(f"  {r['name']:>20}  acc={r['accuracy']:.3f}  "
          f"recall={r['weightedRecall']:.3f}  f1={r['f1']:.3f}")
print("best:", summary["best"]["name"], "->", summary["model_uri"])
print("reloaded-from-registry acc:", round(summary["reload_metrics"]["accuracy"], 3))

Successfully registered model 'barcelona_price_tier'.
Created version '1' of model 'barcelona_price_tier'.
/content/drive/MyDrive/BSE/Big Data/Homework_Labs/Lab3-BigDataArchitectures/lab3-data-engineering/src/training.py:140: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(
2026/06/24 13:21:48 INFO mlflow.spark: URI 'models:/barcelona_price_tier/Production/sparkml' does not point to the current DFS.
2026/06/24 13:21:48 INFO mlflow.spark: File 'models:/barcelona_price_tier/Production/sparkml' not found on DFS. Will attempt to upload the file.


data_fingerprint: 87b1dc170d019a59
   logistic_regression  acc=0.722  recall=0.722  f1=0.724
         random_forest  acc=0.833  recall=0.833  f1=0.833
best: random_forest -> models:/barcelona_price_tier/Production
reloaded-from-registry acc: 0.833


In [7]:
# Copy the MLflow store to Drive so runs + registry survive a Colab disconnect.
if local_store != drive_store:
    shutil.rmtree(drive_store, ignore_errors=True)
    shutil.copytree(local_store, drive_store)
    print("synced MLflow store ->", drive_store)

synced MLflow store -> /content/drive/MyDrive/BSE/Big Data/Homework_Labs/Lab3-BigDataArchitectures/lab3-data-engineering/mlflow
